In [ ]:
import os, sqlite3, warnings
import numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
import plotly.express as px, plotly.graph_objects as go
from pathlib import Path
warnings.filterwarnings("ignore")
BASE_DIR = Path(os.getcwd()).resolve().parent
DB_PATH = BASE_DIR / "data" / "db" / "bluestock_mf.db"
CHARTS_DIR = BASE_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, "figure.figsize": (12, 6)})
print(f"DB exists: {DB_PATH.exists()}")

In [ ]:
RF_DAILY = 0.065 / 252

In [ ]:
def query(sql):
    conn = sqlite3.connect(DB_PATH); df = pd.read_sql(sql, conn); conn.close(); return df

nav = query("SELECT date_id, amfi_code, nav FROM fact_nav ORDER BY date_id")
nav["date"] = pd.to_datetime(nav["date_id"])
pivot = nav.pivot(index="date", columns="amfi_code", values="nav").ffill()
daily_returns = pivot.pct_change().dropna()
txn = query("SELECT investor_id, date_id, amfi_code, transaction_type, amount FROM fact_transactions ORDER BY investor_id, date_id")
txn["date"] = pd.to_datetime(txn["date_id"])
portfolio = query("SELECT amfi_code, sector, weight_pct FROM fact_portfolio")
print("Data loaded")

In [ ]:
var_cvar = {}
for code in daily_returns.columns:
    s = daily_returns[code].dropna()
    var = s.quantile(0.05)
    cvar = s[s <= var].mean()
    var_cvar[code] = {"VaR_95": var, "CVaR_95": cvar}
vc = pd.DataFrame(var_cvar).T.sort_values("VaR_95")
print("Bottom 5 by VaR:"); print(vc.head())
vc.to_csv(BASE_DIR / "var_cvar_report.csv")

In [ ]:
key = daily_returns.columns[:5]
roll = (daily_returns[key].rolling(90).mean() - RF_DAILY) / daily_returns[key].rolling(90).std() * np.sqrt(252)
fig = go.Figure()
for c in key:
    fig.add_trace(go.Scatter(x=roll.index, y=roll[c].dropna(), mode="lines", name=str(c)))
fig.add_hline(y=1, line_dash="dash", line_color="gray", annotation_text="Sharpe=1")
fig.update_layout(title="Rolling 90-Day Sharpe", template="plotly_white")
fig.write_image(str(CHARTS_DIR / "rolling_sharpe.png"), width=1400, height=700, scale=2)
fig.show()

In [ ]:
first = txn.groupby("investor_id")["date"].min().dt.year.reset_index()
first.columns = ["investor_id", "cohort_year"]
txn_m = txn.merge(first, on="investor_id")
cohort = txn_m.groupby(["cohort_year", "investor_id"]).agg(
    total=("amount", "sum"), sip_count=("transaction_type", lambda x: (x == "SIP").sum())
).groupby("cohort_year").mean().reset_index()
print("Cohort summary:"); print(cohort)

In [ ]:
sip = txn[txn["transaction_type"] == "SIP"].sort_values(["investor_id", "date"])
sip["gap"] = sip.groupby("investor_id")["date"].diff().dt.days
summary = sip.groupby("investor_id").agg(
    total_sips=("transaction_type", "count"), max_gap=("gap", "max")
).reset_index()
at_risk = summary[(summary["total_sips"] >= 6) & (summary["max_gap"] > 35)]
eligible = summary[summary["total_sips"] >= 6]
pct = len(at_risk) / len(eligible) * 100 if len(eligible) > 0 else 0
print(f"At-risk: {len(at_risk)} / {len(eligible)} = {pct:.1f}%")

In [ ]:
risk_map = {"Low": ["Low", "Moderately Low"], "Moderate": ["Moderate", "Moderately High"], "High": ["High", "Very High"]}
fund = query("SELECT amfi_code, scheme_name, fund_house, risk_grade FROM dim_fund")
perf = query("SELECT amfi_code, sharpe_ratio, expense_ratio_pct FROM fact_performance")
merged = fund.merge(perf, on="amfi_code")
for appetite in ["Low", "Moderate", "High"]:
    filtered = merged[merged["risk_grade"].isin(risk_map[appetite])]
    top = filtered.nlargest(3, "sharpe_ratio")
    print(f"\n{appetite}:")
    print(top[["scheme_name", "fund_house", "sharpe_ratio"]].to_string(index=False))

In [ ]:
hhi = portfolio.groupby("amfi_code").apply(lambda g: ((g["weight_pct"] / 100) ** 2).sum()).rename("HHI").reset_index()
hhi = hhi.merge(fund[["amfi_code", "scheme_name", "category"]], on="amfi_code")
concern = hhi[hhi["HHI"] > 0.25]
print(f"Highly concentrated (HHI > 0.25): {len(concern)}")
print(concern[["scheme_name", "HHI", "category"]])